## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- FAISS의 특징과 Chroma와의 차이를 설명할 수 있어요
- FAISS VectorStore를 생성하고 디스크에 저장/로드할 수 있어요
- MMR 검색과 Retriever를 활용해 RAG 체인에 통합할 수 있어요

## 사전 지식

- 노트북 01(Chroma) 학습 완료
- OpenAI API 키 설정

# FAISS VectorStore

FAISS(Facebook AI Similarity Search)는 Facebook AI Research에서 개발한 고속 벡터 검색 라이브러리예요. 메모리 기반 인덱싱으로 Chroma보다 빠른 검색 속도를 제공해요.

## Chroma vs FAISS 비교

```mermaid
flowchart LR
    subgraph Chroma["Chroma"]
        CD["디스크 기반<br/>영구 저장"]:::storage
        CM["메타데이터<br/>풍부한 필터"]:::process
        CS["소~중규모<br/>(수만 문서)"]:::output
    end

    subgraph FAISS["FAISS"]
        FD["메모리 기반<br/>save_local/load_local"]:::storage
        FM["고속 검색<br/>5~10배 빠름"]:::process
        FS["중~대규모<br/>(수백만 문서)"]:::output
    end

    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

> 🔑 **핵심 개념**: FAISS의 핵심 차별점은 **ANN(Approximate Nearest Neighbor)** 검색이에요. 정확한 최근접 이웃을 찾는 대신 근사값을 빠르게 찾아서, 수백만 벡터에서도 밀리초 단위 검색이 가능해요.

**FAISS를 선택하는 경우**:
- 문서가 수만 개를 넘어 대규모 검색이 필요할 때
- 밀리초 단위의 빠른 응답 시간이 필요할 때
- RAM에 데이터를 모두 로드할 수 있는 환경

**참고 문서**: [FAISS 공식 문서](https://faiss.ai/) | [LangChain FAISS 통합](https://python.langchain.com/docs/integrations/vectorstores/faiss/)

## 환경 설정

In [1]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

---

## 1. 데이터 준비

In [2]:

# ---------------------------------------------------
# 문서 로드 및 분할 (데이터 준비)
# ---------------------------------------------------

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)

# 문서 로드 및 분할 (이미 로드된 경우 재사용)
if "docs_nlp" not in globals():
    loader_nlp = TextLoader("data/nlp-keywords.txt")
    docs_nlp = loader_nlp.load_and_split(text_splitter)
else:
    loader_nlp = None

if "docs_finance" not in globals():
    loader_finance = TextLoader("data/finance-keywords.txt")
    docs_finance = loader_finance.load_and_split(text_splitter)

print(f"NLP 문서: {len(docs_nlp)}개")
print(f"금융 문서: {len(docs_finance)}개")


NLP 문서: 11개
금융 문서: 6개


---

## 2. FAISS VectorStore 생성

### 2.1 from_documents로 생성

Chroma와 사용법이 거의 동일해요. `from_documents()`로 간단하게 생성할 수 있어요.

In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---------------------------------------------------
# FAISS VectorStore 생성
# ---------------------------------------------------

# ============================================================
# TODO: FAISS.from_documents()로 벡터 저장소를 생성해보세요
# 힌트: FAISS.from_documents(documents, embedding)
# 예상 결과: "FAISS VectorStore가 생성되었습니다." 출력
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 필요한 경우 문서를 먼저 준비
if "docs_nlp" not in globals():
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)
    loader_nlp = TextLoader("data/nlp-keywords.txt")
    docs_nlp = loader_nlp.load_and_split(text_splitter)

# 1단계: FAISS VectorStore 생성
# FAISS: 메모리 기반 고속 벡터 인덱싱 라이브러리 (Facebook AI Research 개발)
vectorstore = FAISS.from_documents(
    documents=docs_nlp,
    embedding=OpenAIEmbeddings()
)

print("✅ FAISS VectorStore가 생성되었습니다.")


✅ FAISS VectorStore가 생성되었습니다.


---

## 3. 유사도 검색

FAISS의 검색 API는 Chroma와 동일해요. 같은 코드를 두 VectorStore에서 바꿔서 쓸 수 있어요.

> 🎯 **강의 포인트**: LangChain의 VectorStore들은 **공통 인터페이스**를 공유해요. `similarity_search()`, `as_retriever()` 등 주요 메서드의 이름과 시그니처가 동일해서, Chroma에서 FAISS로, 또는 Pinecone으로 교체해도 RAG 체인 코드를 거의 수정하지 않아도 돼요. 이것이 LangChain 추상화의 핵심 가치예요.

In [4]:

# ---------------------------------------------------
# 기본 유사도 검색 (k=4 기본값)
# ---------------------------------------------------

# similarity_search(): Chroma와 동일한 API — VectorStore 교체 시 코드 변경 최소화
results = vectorstore.similarity_search("Word2Vec과 임베딩 기법에 대해 설명해줘")

print("=== 검색 결과 ===")
for idx, doc in enumerate(results, 1):
    print(f"\n[{idx}] {doc.page_content[:150]}...")


=== 검색 결과 ===

[1] 정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워...

[2] 정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니다.
예시: HuggingFace의 Transformers 라이브러리를 사용하여 감정 분석...

[3] Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환...

[4] 정의: 토크나이저는 텍스트 데이터를 토큰으로 분할하는 도구입니다. 이는 자연어 처리에서 데이터를 전처리하는 데 사용됩니다.
예시: "I love programming."이라는 문장을 ["I", "love", "programming", "."]으로 분할합니다.
연관키워드...


### 3.1 k 값 조정

In [5]:

# ---------------------------------------------------
# k 파라미터로 검색 결과 개수 조정
# ---------------------------------------------------

# k=2: 상위 2개 결과만 반환
results_top2 = vectorstore.similarity_search("Word2Vec과 임베딩 기법에 대해 설명해줘", k=2)

print(f"검색된 문서: {len(results_top2)}개\n")
for idx, doc in enumerate(results_top2, 1):
    print(f"{idx}. {doc.page_content[:100]}...")


검색된 문서: 2개

1. 정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: W...
2. 정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니...


---

## 4. 디스크 저장 및 로드

FAISS는 `save_local()`과 `load_local()` 메서드로 인덱스를 저장하고 불러올 수 있어요. Chroma의 `persist_directory`와 같은 역할이에요.

> ⚠️ **자주 하는 실수**: `load_local()` 시 `allow_dangerous_deserialization=True`가 필요해요. 신뢰할 수 있는 자신이 저장한 파일만 로드해야 해요. 외부에서 받은 FAISS 인덱스 파일은 임의 코드를 실행할 수 있는 pickle 파일을 포함하므로 주의해야 해요.

In [6]:
# ---------------------------------------------------
# FAISS 인덱스 디스크 저장 및 로드
# ---------------------------------------------------

# ============================================================
# TODO: save_local()로 저장하고 load_local()로 다시 불러와보세요
# 힌트: vectorstore.save_local(경로) / FAISS.load_local(경로, embeddings, allow_dangerous_deserialization=True)
# 예상 결과: 저장 완료 및 로드 완료 메시지 출력
# ============================================================

# 1단계: 디스크에 저장
# save_local(): FAISS 인덱스 파일(.faiss)과 메타데이터(.pkl)를 저장
FAISS_PATH = "./faiss_nlp_index"
vectorstore.save_local(FAISS_PATH)
print(f"✅ FAISS 인덱스가 '{FAISS_PATH}'에 저장되었습니다.")

# 2단계: 디스크에서 로드
# allow_dangerous_deserialization=True: pickle 역직렬화 허용 (신뢰 파일에만 사용)
loaded_vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=OpenAIEmbeddings(),
    allow_dangerous_deserialization=True  # 저장된 파일 역직렬화 허용
)
print("✅ FAISS 인덱스가 로드되었습니다.")


✅ FAISS 인덱스가 './faiss_nlp_index'에 저장되었습니다.
✅ FAISS 인덱스가 로드되었습니다.


---

## 5. MMR 검색

FAISS도 MMR(Maximum Marginal Relevance) 검색을 지원해요. 검색 결과의 다양성을 높이고 싶을 때 사용해요.

> 💡 **실무 팁**: FAISS에서는 `as_retriever(search_type="mmr")`와 `max_marginal_relevance_search()` 두 가지 방법으로 MMR을 사용할 수 있어요. 체인에 연결할 때는 `as_retriever()` 방식이 더 편리해요.

In [7]:
# ---------------------------------------------------
# MMR 검색 (다양성을 고려한 결과 반환)
# ---------------------------------------------------

# ============================================================
# TODO: max_marginal_relevance_search()로 MMR 검색을 수행해보세요
# 힌트: loaded_vectorstore.max_marginal_relevance_search(query, k=4, fetch_k=20)
# 예상 결과: 유사한 문서들이 아닌 다양한 주제의 4개 문서 반환
# ============================================================

# max_marginal_relevance_search(): FAISS의 직접 MMR 메서드
# fetch_k: 1차 유사도 검색으로 후보 선정
# k: 최종 반환 개수 (다양성 고려 후 선택)
mmr_results = loaded_vectorstore.max_marginal_relevance_search(
    query="자연어 처리와 딥러닝 기술",
    k=4,
    fetch_k=20
)

print("=== MMR 검색 결과 (다양성 고려) ===")
for idx, doc in enumerate(mmr_results, 1):
    print(f"\n[{idx}] {doc.page_content[:120]}...")


=== MMR 검색 결과 (다양성 고려) ===

[1] 정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니다.
예시: HuggingFace의 ...

[2] 정의: 멀티모달은 여러 종류의 데이터 모드(예: 텍스트, 이미지, 소리 등)를 결합하여 처리하는 기술입니다. 이는 서로 다른 형식의 데이터 간의 상호 작용을 통해 보다 풍부하고 정확한 정보를 추출하거나 예측하는 데 ...

[3] 정의: DataFrame은 행과 열로 이루어진 테이블 형태의 데이터 구조로, 주로 데이터 분석 및 처리에 사용됩니다.
예시: 판다스 라이브러리에서 DataFrame은 다양한 데이터 타입의 열을 가질 수 있으며, 데이...

[4] GPT (Generative Pretrained Transformer)

정의: GPT는 대규모의 데이터셋으로 사전 훈련된 생성적 언어 모델로, 다양한 텍스트 기반 작업에 활용됩니다. 이는 입력된 텍스트에 기반하여 ...


---

## 6. Retriever로 변환

`as_retriever()`로 LangChain 체인에서 사용할 수 있는 Retriever로 변환해요.

In [8]:
# ---------------------------------------------------
# FAISS VectorStore를 Retriever로 변환
# ---------------------------------------------------

# ============================================================
# TODO: as_retriever()로 Retriever를 만들고 검색을 실행해보세요
# 힌트: loaded_vectorstore.as_retriever(search_kwargs={"k": 3})
# 예상 결과: 트랜스포머/어텐션 관련 문서 3개 반환
# ============================================================

# 1단계: k=3으로 Retriever 생성
# search_kwargs: 검색 파라미터를 딕셔너리로 전달
retriever = loaded_vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved_docs = retriever.invoke("트랜스포머와 어텐션 메커니즘")

print(f"검색된 문서: {len(retrieved_docs)}개\n")
for idx, doc in enumerate(retrieved_docs, 1):
    print(f"{idx}. {doc.page_content[:100]}...")


검색된 문서: 3개

1. 정의: 멀티모달은 여러 종류의 데이터 모드(예: 텍스트, 이미지, 소리 등)를 결합하여 처리하는 기술입니다. 이는 서로 다른 형식의 데이터 간의 상호 작용을 통해 보다 풍부하고 정...
2. 정의: 토크나이저는 텍스트 데이터를 토큰으로 분할하는 도구입니다. 이는 자연어 처리에서 데이터를 전처리하는 데 사용됩니다.
예시: "I love programming."이라는 문장...
3. 정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니...


### 6.1 MMR Retriever

In [9]:

# ---------------------------------------------------
# MMR 방식 Retriever 설정
# ---------------------------------------------------

# search_type="mmr": as_retriever()에서도 MMR 방식 사용 가능
mmr_retriever = loaded_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10  # 10개 후보에서 다양성 고려해 3개 선택
    }
)

mmr_docs = mmr_retriever.invoke("머신러닝과 데이터 처리")

print("=== MMR Retriever 결과 ===")
for idx, doc in enumerate(mmr_docs, 1):
    print(f"\n[{idx}] {doc.page_content[:100]}...")


=== MMR Retriever 결과 ===

[1] 정의: 멀티모달은 여러 종류의 데이터 모드(예: 텍스트, 이미지, 소리 등)를 결합하여 처리하는 기술입니다. 이는 서로 다른 형식의 데이터 간의 상호 작용을 통해 보다 풍부하고 정...

[2] 정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.
예시: ...

[3] 정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니...


---

## 마무리

### Chroma vs FAISS 한눈에 비교

| 특징 | Chroma | FAISS |
|------|--------|-------|
| 저장 방식 | 디스크 영구 저장 | 메모리 (저장/로드 가능) |
| 검색 속도 | 중간 | Chroma 대비 5~10배 빠름 |
| 메타데이터 필터 | 풍부한 지원 | 제한적 |
| 적합한 규모 | 소~중규모 | 중~대규모 |
| 설치 | `pip install chromadb` | `pip install faiss-cpu` |

**선택 기준**: 수만 개 이하의 문서는 Chroma, 그 이상이거나 검색 속도가 중요하면 FAISS를 사용해보세요.

### 다음 학습

**6-4 노트북 03**: Pinecone — 인프라 관리 없이 클라우드에서 운영하는 완전 관리형 벡터 DB를 배워볼게요.